[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/electromagnetism/constitutive/constitutive.ipynb)

# Electromagnetic Constitutive Maps, Birefringence & Fresnel Drag in Spacetime Algebra

In a material, light is carried by two fields: the electromagnetic field and the excitation the material answers it with, both bivectors of spacetime. The material is the map between them, `Bivector <- Bivector`, written in numga by leaving a bivector open in an expression. This notebook builds glass, a birefringent crystal and a magnetic ferrite this way, finds the speeds at which light travels through each from where a wave map loses rank, and sets the glass in motion to watch light dragged along with it.

In the tensor formulation of electrodynamics in media the field and the excitation read as $F_{\mu\nu}$ and $G^{\mu\nu}$, and the material as the constitutive tensor, $G^{\mu\nu} = \tfrac12 \chi^{\mu\nu\rho\sigma} F_{\rho\sigma}$.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt

from numga import NumpyContext
from numga.algebras import STA
from examples.electromagnetism.constitutive import render

# The spacetime algebra, with signature (+, -, -, -).
ctx = NumpyContext(STA)
mv = ctx.multivector

Scalar = STA.gatype.scalar()
Vector = STA.gatype.vector()
# Fields and excitations are bivectors.
Bivector = STA.gatype.bivector()
# Polarizations are spatial vectors, orthogonal to the observer's time axis.
Spatial = STA.subspace("x y z")
# Rotations and boosts.
Rotor = STA.gatype.rotor()

# A material, from fields to excitations, and a map on vectors.
Constitutive = STA.gatype((Bivector, Bivector))  # Bivector <- Bivector
Permittivity = STA.gatype((Vector, Vector))      # Vector <- Vector

t, x, y, z = mv.vector(np.eye(4))

## 1. The Observer, Media & Extensor Constitutive Relations

An observer moving along the time axis `t` splits a field into an electric and a magnetic part. `Bivector.commutator(t)` takes the electric vector out of a field, and `.wedge(t)` puts it back as a bivector: with the field left open, `electric` is the map that keeps a field's electric part, and `Bivector - electric` keeps the magnetic part. Glass answers each part with its own factor. A crystal answers the electric vector differently along its axes, with a map on vectors between taking the electric vector out and putting it back; a ferrite does the same to the magnetic vector, which the dual of the field brings to the electric side.

In the three-dimensional vector notation of Maxwell's equations the materials read as $\mathbf{D} = \boldsymbol{\epsilon}\,\mathbf{E}$ and $\mathbf{H} = \boldsymbol{\mu}^{-1}\mathbf{B}$, and in tensor index notation the electric part as $E_\mu = F_{\mu\nu} t^\nu$.

In [ ]:
# The observer's split: Bivector.commutator(t) takes a field's electric vector and .wedge(t) returns
# it to a bivector, so with the field open this is the map that keeps the electric part. The rest of
# the field is the magnetic part:
electric = Bivector.commutator(t).wedge(t)                                      # [] Bivector <- Bivector
magnetic = Bivector - electric                                                  # [] Bivector <- Bivector

# Glass answers the electric part with its permittivity and the magnetic part with one over its
# permeability; maps scale and add:
eps, mu = 2.25, 1.0
glass = eps * electric + (1.0 / mu) * magnetic                                  # [] Bivector <- Bivector

# A crystal answers the electric vector differently along its axes. Spatial axes square to minus one
# in this signature, so -x * (x | Vector) keeps a vector's part along x; weighted and summed over the
# axes it is the permittivity, a map on vectors. It sits between taking the electric vector out of
# the field and returning it; the magnetic part is answered as in glass:
permittivity = -(
    2.25 * x * (x | Vector) +
    1.5 * y * (y | Vector) +
    1.5 * z * (z | Vector)
)                                                                               # [] Vector <- Vector
crystal = permittivity(Bivector.commutator(t)).wedge(t) + (1.0 / mu) * magnetic # [] Bivector <- Bivector

# A ferrite answers the magnetic vector differently along its axes. The dual of a field swaps its
# electric and magnetic parts, so Bivector.dual().commutator(t) takes the magnetic vector; the
# inverse permeability acts on it, .wedge(t) returns it to a bivector, and .dual_inverse() carries
# that back to the magnetic side. The electric part is answered as in glass:
permeability_inv = -(
    1.0 * x * (x | Vector) +
    0.5 * y * (y | Vector) +
    1.0 * z * (z | Vector)
)                                                                               # [] Vector <- Vector
ferrite = (
    eps * electric +
    permeability_inv(Bivector.dual().commutator(t)).wedge(t).dual_inverse()
)                                                                               # [] Bivector <- Bivector

print(electric)

## 2. The 1D Dispersion Scan: Detecting Waves via Singular Values

A plane wave's field is its wave vector wedged with its polarization. The material answers with an excitation, and Maxwell's equations without sources ask that excitation's inner product with the wave vector to vanish. With the polarization left open, `k.commutator(medium(k.wedge(Spatial)))` is the wave map, from polarizations to vectors, and a wave travels exactly where it loses rank: where its smallest singular value drops to zero. `k = speeds * t + z` tries a batch of phase speeds along z at once, and the dips mark the speeds at which light travels.

In the conventional notation of the spacetime algebra the wave vector reads as $k = \omega t + \mathbf{k}$, the field as $F = k \wedge a$ and the wave condition as $k \cdot \chi(k \wedge a) = 0$; in the tensor formulation, $k_\mu \chi^{\mu\nu\rho\sigma} k_\rho a_\sigma = 0$, whose vanishing determinant is the Fresnel equation of the medium.

In [ ]:
# Trial phase speeds, and a wave vector for each, travelling along z:
speeds = np.linspace(0.05, 1.5, 3001)
k = speeds * t + z                                                            # [n_speeds] Vector

# The wave map with the polarization open: k.wedge(Spatial) is the wave's field, medium(...) the
# material's answer, and k.commutator(...) what Maxwell's equations ask to vanish. A wave travels
# where the smallest singular value drops to zero:
media = {
    "glass at rest": glass,
    "crystal along z": crystal,
    "ferrite along z": ferrite,
}

curves = {}
for name, medium in media.items():
    wave = k.commutator(medium(k.wedge(Spatial)))                             # [n_speeds] Vector <- Spatial
    curves[name] = wave.svdvals()[..., -1]

refractive_index = np.sqrt(eps * mu)
expected = {
    "glass at rest": [1.0 / refractive_index],
    "crystal along z": [1.0 / np.sqrt(2.25), 1.0 / np.sqrt(1.5)],
    "ferrite along z": [1.0 / np.sqrt(2.25 * 2.0), 1.0 / np.sqrt(2.25)],
}

# The smallest singular value against the trial speed, with the expected speeds marked:
fig = render.draw_dispersion_figure(speeds, curves, expected)
plt.show()

## 3. Physical Eigenmodes: Transverse Polarizations & Birefringence

At each dip, which polarization travels? `wave.svd()` at an allowed speed returns, as its last right singular vector, the polarization the map loses: the wave that travels at that speed. Along z the crystal's slow and fast waves are polarized along its axes x and y. In glass both travel at one speed, so a linear polarization stays linear; in the crystal the fast wave pulls ahead, and a polarization between the axes turns elliptical and back as it travels.

In matrix notation the polarization reads as the null vector $a \in \mathbb{R}^3$ of the wave matrix, and the phase slip between the axes as $(k_x - k_y)\,z$.

In [ ]:
# The crystal's allowed speeds, where its smallest singular value dips:
crystal_svals = curves["crystal along z"]
left, mid, right = crystal_svals[:-2], crystal_svals[1:-1], crystal_svals[2:]
is_resonance_dip = (mid <= left) & (mid <= right) & (mid < 2e-3)
resonant_speeds = speeds[1:-1][is_resonance_dip]
v_slow, v_fast = float(resonant_speeds[0]), float(resonant_speeds[1])

# The wave map at each allowed speed, and the polarization it loses: its last right singular vector.
k_slow = v_slow * t + z                                                       # [] Vector
wave_slow = k_slow.commutator(crystal(k_slow.wedge(Spatial)))                 # [] Vector <- Spatial
left_slow, values_slow, right_slow = wave_slow.svd()
pol_slow = right_slow[-1]                                                     # [] Spatial

k_fast = v_fast * t + z                                                       # [] Vector
wave_fast = k_fast.commutator(crystal(k_fast.wedge(Spatial)))                 # [] Vector <- Spatial
left_fast, values_fast, right_fast = wave_fast.svd()
pol_fast = right_fast[-1]                                                     # [] Spatial

# Glass carries both polarizations at one speed; the crystal carries each at its own:
glass_modes = [(v_slow, pol_slow), (v_slow, pol_fast)]
crystal_modes = [(v_slow, pol_slow), (v_fast, pol_fast)]
fig = render.draw_wave_comparison_figure(glass_modes, crystal_modes)
plt.show()

## 4. 2D Directional Anisotropy: Polar Fresnel Wave Surfaces

Light does not only travel along z. Turning the heading through the xz plane and trying every speed along every heading in one batch, the dips trace the speed of light against direction. In glass it is one circle. In the crystal it splits into two sheets, a circle and an ellipse that touch on the optic axis.

In the notation of crystal optics this curve reads as the Fresnel wave normal surface $v(\theta)$ for headings $\sin\theta\, \mathbf{x} + \cos\theta\, \mathbf{z}$.

In [ ]:
# Headings through the xz plane, and every trial speed along each:
angles = np.linspace(0, 2 * np.pi, 90)
sweep_speeds = np.linspace(0.4, 1.0, 601)
k_dirs = np.sin(angles) * x + np.cos(angles) * z                              # [n_angles] Vector
k_grid = sweep_speeds[:, None] * t + k_dirs[None, :]                          # [n_speeds, n_angles] Vector

# The wave maps over the whole grid in one batch:
w_glass = k_grid.commutator(glass(k_grid.wedge(Spatial)))                     # [n_speeds, n_angles] Vector <- Spatial
w_crystal = k_grid.commutator(crystal(k_grid.wedge(Spatial)))                 # [n_speeds, n_angles] Vector <- Spatial

# The speeds at which light travels, against its heading:
fig = render.draw_fresnel_surface_figure(
    angles,
    {
        "glass (rest)": w_glass.svdvals()[..., -1],
        "crystal": w_crystal.svdvals()[..., -1],
    },
    speeds=sweep_speeds,
)
plt.show()

## 5. Spacetime Dynamics: Relativistic Boost & Fresnel Drag

Setting a material in motion moves its map like any other: `boost >> glass(boost << Bivector)` pulls the field into the glass's rest frame, lets the glass answer, and pushes the answer back out. The boost is a rotor, the exponential of half the rapidity times `z ^ t`, batched over the glass's speeds. Light travelling with the moving glass is dragged along, and against it held back. The plot marks the speeds where the moving glass's wave map loses rank, against the exact relativistic sum of the speed of light in glass and the glass's speed (solid) and Fresnel's first-order drag (dashed), which departs from it beyond about a third of the speed of light.

In tensor index notation moving the material transforms each of the constitutive tensor's four indices, $\chi'^{\alpha\beta}{}_{\gamma\delta} = \Lambda^\alpha{}_\mu \Lambda^\beta{}_\nu\, \chi^{\mu\nu}{}_{\rho\sigma}\, (\Lambda^{-1})^\rho{}_\gamma (\Lambda^{-1})^\sigma{}_\delta$; in the notation of special relativity the sum reads as $v = \frac{c/n \pm \beta}{1 \pm \beta/n}$, and Fresnel's drag as $v \approx c/n \pm \beta(1 - 1/n^2)$.

In [ ]:
# Speeds of the glass, as fractions of the speed of light, and trial speeds of light:
betas = np.linspace(-0.6, 0.6, 17)
drag_speeds = np.linspace(0.05, 1.2, 1151)

# A rotor moves a vector by its sandwich, rotor >> v. It moves a map by pulling the input into the
# glass's rest frame, rotor << Bivector, and pushing the glass's answer back out:
rapidity = np.arctanh(betas)
boost_rotors = ((z ^ t) * (rapidity / 2.0)).exp()                             # [n_betas] Rotor
moving_media = boost_rotors >> glass(boost_rotors << Bivector)                # [n_betas] Bivector <- Bivector

# Waves along the motion and against it, over every speed of light and of the glass in one batch:
k_down = drag_speeds[:, None] * t + z                                         # [n_speeds, n_betas] Vector
k_up = drag_speeds[:, None] * t - z                                           # [n_speeds, n_betas] Vector

w_down = k_down.commutator(moving_media(k_down.wedge(Spatial)))               # [n_speeds, n_betas] Vector <- Spatial
w_up = k_up.commutator(moving_media(k_up.wedge(Spatial)))                     # [n_speeds, n_betas] Vector <- Spatial

v_down = drag_speeds[w_down.svdvals()[..., -1].argmin(axis=0)]
v_up = drag_speeds[w_up.svdvals()[..., -1].argmin(axis=0)]

# The speed of light in the moving glass, against the relativistic sum and Fresnel's drag:
fig = render.draw_fresnel_drag_figure(betas, v_down, v_up, eps=eps, mu=mu)
plt.show()